# Notebook 01: Cruise industry passenger growth (1990–2025)

**Research context:** *Rethinking Vacations in an Era of Immobility*  
Data sources: CLIA State of the Cruise Industry 2025; Port Economics, Management and Policy (Notteboom et al., 2026); ScienceDirect seasonality study.

This notebook documents the exponential growth of the global cruise industry and visualizes the post-COVID recovery — which not only restored but surpassed pre-pandemic levels. This data supports the argument that tourism degrowth cannot be achieved without directly confronting the structural growth imperative of tourism capitalism (Fletcher et al., 2019).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# Paths
DATA = Path('../data')
FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)

# Color palette (consistent across all notebooks)
BLUE = '#185FA5'
BLUE_LIGHT = '#B5D4F4'
RED = '#E24B4A'
RED_LIGHT = '#F7C1C1'
GRAY = '#888780'
GRAY_LIGHT = '#D3D1C7'
AMBER = '#BA7517'
TEXT = '#2C2C2A'
TEXT_SEC = '#5F5E5A'
BG = '#FAFAF8'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.alpha': 0.4,
    'grid.linewidth': 0.5,
    'figure.facecolor': BG,
    'axes.facecolor': BG,
})

print('Setup complete.')

In [ ]:
df = pd.read_csv(DATA / 'passengers.csv')
print(df.shape)
df.head(10)

In [ ]:
# --- Summary stats ---
growth_pct = (df.loc[df.year==2024, 'passengers_millions'].values[0] /
              df.loc[df.year==1990, 'passengers_millions'].values[0] - 1) * 100
print(f'Growth 1990–2024: +{growth_pct:.0f}%')
print(f'Peak pre-COVID (2019): {df.loc[df.year==2019, "passengers_millions"].values[0]}M')
print(f'COVID trough (2020):   {df.loc[df.year==2020, "passengers_millions"].values[0]}M')
print(f'2023 vs 2019:          {df.loc[df.year==2023, "passengers_millions"].values[0]/df.loc[df.year==2019, "passengers_millions"].values[0]*100:.0f}%')

In [ ]:
# --- Matplotlib static figure (high-res for PDF) ---
covid_years = [2020, 2021]
colors = [RED if y in covid_years else BLUE for y in df.year]
alphas = [0.85 if y in covid_years else 0.7 for y in df.year]
edge_colors = [RED if y in covid_years else BLUE for y in df.year]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(df.year, df.passengers_millions,
              color=colors, alpha=0.75, edgecolor=edge_colors,
              linewidth=0.8, width=0.75)

# Annotate key moments
annotations = {
    2019: ('Pre-COVID\npeak: 28.5M', -2.2),
    2020: ('COVID\nshutdown', 0.4),
    2023: ('31.7M\n(107% of 2019)', 0.4),
}
for yr, (label, offset) in annotations.items():
    val = df.loc[df.year==yr, 'passengers_millions'].values[0]
    ax.annotate(label,
                xy=(yr, val + offset),
                ha='center', va='bottom' if offset > 0 else 'top',
                fontsize=9, color=TEXT_SEC,
                arrowprops=dict(arrowstyle='-', color=GRAY_LIGHT, lw=0.8) if offset < 0 else None)

# Legend
legend_items = [
    mpatches.Patch(color=BLUE, alpha=0.75, label='Cruise passengers (millions)'),
    mpatches.Patch(color=RED, alpha=0.75, label='COVID-19 disruption'),
]
ax.legend(handles=legend_items, loc='upper left', frameon=False, fontsize=10)

ax.set_xlabel('Year', color=TEXT_SEC, labelpad=8)
ax.set_ylabel('Passengers (millions)', color=TEXT_SEC, labelpad=8)
ax.tick_params(colors=TEXT_SEC, labelsize=10)
ax.set_xlim(1989, 2026)
ax.set_ylim(0, 42)
ax.spines['bottom'].set_color(GRAY_LIGHT)

fig.suptitle('Global cruise passengers, 1990–2025',
             x=0.08, y=1.01, ha='left', fontsize=14, fontweight='500', color=TEXT)
ax.set_title('Growth of +810% over 35 years — faster than any other tourism sector',
             loc='left', fontsize=10, color=TEXT_SEC, pad=6)

fig.text(0.08, -0.04,
         'Sources: CLIA State of the Cruise Industry 2025; Port Economics, Management and Policy (Notteboom et al., 2026)',
         fontsize=8, color=GRAY, ha='left')

plt.tight_layout()
fig.savefig(FIGURES / 'fig_growth.png', dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved: figures/fig_growth.png')

In [ ]:
# --- Plotly interactive version ---
covid_mask = df.year.isin([2020, 2021])

fig_p = go.Figure()

fig_p.add_trace(go.Bar(
    x=df.loc[~covid_mask, 'year'],
    y=df.loc[~covid_mask, 'passengers_millions'],
    name='Cruise passengers (millions)',
    marker_color=BLUE, marker_opacity=0.75,
    hovertemplate='%{x}: %{y}M passengers<extra></extra>'
))

fig_p.add_trace(go.Bar(
    x=df.loc[covid_mask, 'year'],
    y=df.loc[covid_mask, 'passengers_millions'],
    name='COVID-19 disruption',
    marker_color=RED, marker_opacity=0.8,
    hovertemplate='%{x}: %{y}M passengers (COVID period)<extra></extra>'
))

fig_p.update_layout(
    title=dict(text='Global cruise passengers, 1990–2025', font_size=16, x=0, xanchor='left'),
    xaxis_title='Year',
    yaxis_title='Passengers (millions)',
    barmode='overlay',
    plot_bgcolor=BG,
    paper_bgcolor=BG,
    font_color=TEXT,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(t=80, b=60),
    annotations=[dict(
        text='Sources: CLIA 2025; Notteboom et al. 2026',
        xref='paper', yref='paper', x=0, y=-0.12,
        showarrow=False, font_size=9, font_color=GRAY
    )]
)
fig_p.update_yaxes(gridcolor=GRAY_LIGHT, gridwidth=0.5, zeroline=False)
fig_p.update_xaxes(showgrid=False)

fig_p.write_html(FIGURES / 'fig_growth.html')
fig_p.show()
print('Saved: figures/fig_growth.html')